# Feature Engineering: SPECTER + Venue + Author

Combines SPECTER2 embeddings (768-dim) with venue prestige + author features.
Uses the **same temporal split** as nb24 (indices from X_train/test_temporal.pkl).

**Prerequisites**: run `20c_specter_embeddings.ipynb` first → `specter_features.pkl`

In [ ]:
import sys
sys.path.append('../')

import pandas as pd
import numpy as np
from pathlib import Path
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold, cross_val_score, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import matplotlib.pyplot as plt

feature_dir = Path('../data/features')
SEED = 42
print('Imports OK')

## 1. Load SPECTER + Venue + Author Features

In [ ]:
specter  = pd.read_pickle(feature_dir / 'specter_features.pkl')
venue    = pd.read_pickle(feature_dir / 'venue_features.pkl')
author   = pd.read_pickle(feature_dir / 'author_features.pkl')

print(f'SPECTER : {specter.shape}')
print(f'Venue   : {venue.shape}')
print(f'Author  : {author.shape}')

# Align all on common index
common_idx = specter.index.intersection(venue.index).intersection(author.index)
print(f'\nCommon papers: {len(common_idx)}')

X_all = pd.concat([
    specter.loc[common_idx],
    venue.loc[common_idx],
    author.loc[common_idx],
], axis=1)

X_all = X_all.fillna(X_all.median())
print(f'Combined: {X_all.shape}  (vs TF-IDF baseline: N×5019)')

## 2. Reproduce Temporal Split Using nb24 Indices

In [ ]:
# Load the existing temporal split targets — their index IS the split
y_train = pd.read_pickle(feature_dir / 'y_train_cls_temporal.pkl')
y_test  = pd.read_pickle(feature_dir / 'y_test_cls_temporal.pkl')

train_idx = y_train.index.intersection(common_idx)
test_idx  = y_test.index.intersection(common_idx)

X_train = X_all.loc[train_idx]
X_test  = X_all.loc[test_idx]
y_train = y_train.loc[train_idx]
y_test  = y_test.loc[test_idx]

print(f'Train: {X_train.shape}  positive rate: {y_train.mean()*100:.1f}%')
print(f'Test : {X_test.shape}   positive rate: {y_test.mean()*100:.1f}%')

## 3. Train Models

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
pos_weight = (1 - y_train.mean()) / y_train.mean()

def threshold_sweep(probs, y_true, step=0.005):
    best_f1, best_t = 0, 0.5
    for t in np.arange(0.20, 0.70, step):
        f = f1_score(y_true, probs >= t)
        if f > best_f1:
            best_f1, best_t = f, t
    return best_f1, best_t

models = {
    'LR': LogisticRegression(
        max_iter=3000, solver='saga', class_weight='balanced',
        random_state=SEED, n_jobs=-1
    ),
    'XGBoost': XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
        scale_pos_weight=pos_weight,
        random_state=SEED, n_jobs=-1, verbosity=0, eval_metric='logloss'
    ),
    'LightGBM': LGBMClassifier(
        n_estimators=500, num_leaves=31, learning_rate=0.05,
        min_child_samples=30, subsample=0.8, colsample_bytree=0.8,
        reg_lambda=1.0, class_weight='balanced',
        random_state=SEED, n_jobs=-1, verbose=-1
    ),
}

results = {}
trained = {}
for name, model in models.items():
    print(f'Training {name}...')
    cv_f1 = cross_val_score(model, X_train, y_train, cv=cv, scoring='f1').mean()
    model.fit(X_train, y_train)
    probs = model.predict_proba(X_test)[:, 1]
    test_f1, best_t = threshold_sweep(probs, y_test)
    auc = roc_auc_score(y_test, probs)
    results[name] = dict(cv_f1=cv_f1, test_f1=test_f1, threshold=best_t, auc=auc)
    trained[name] = (model, probs)
    print(f'  CV F1={cv_f1:.4f}  Test F1={test_f1:.4f} (t={best_t:.3f})  AUC={auc:.4f}')

## 4. Tune Best Model on SPECTER Features (scoring=f1)

In [ ]:
print('Running RandomizedSearchCV on XGBoost (scoring=f1, 50 iters)...')

param_dist = {
    'n_estimators':     [200, 300, 500, 700],
    'max_depth':        [3, 4, 5, 6],
    'learning_rate':    [0.01, 0.03, 0.05, 0.08],
    'subsample':        [0.6, 0.7, 0.8, 0.9],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9],
    'min_child_weight': [1, 3, 5, 10],
    'reg_lambda':       [0.5, 1.0, 2.0, 5.0],
    'gamma':            [0, 0.1, 0.3],
}

search = RandomizedSearchCV(
    XGBClassifier(scale_pos_weight=pos_weight, random_state=SEED,
                  n_jobs=-1, verbosity=0, eval_metric='logloss'),
    param_dist, n_iter=50, scoring='f1',
    cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
    n_jobs=-1, random_state=SEED, verbose=1
)
search.fit(X_train, y_train)

probs_tuned = search.best_estimator_.predict_proba(X_test)[:, 1]
tuned_f1, tuned_t = threshold_sweep(probs_tuned, y_test)
tuned_auc = roc_auc_score(y_test, probs_tuned)

results['XGBoost (tuned)'] = dict(
    cv_f1=search.best_score_, test_f1=tuned_f1,
    threshold=tuned_t, auc=tuned_auc
)
print(f'\nBest CV F1 : {search.best_score_:.4f}')
print(f'Test F1    : {tuned_f1:.4f} (threshold={tuned_t:.3f})')
print(f'AUC        : {tuned_auc:.4f}')
print(f'Best params: {search.best_params_}')

## 5. Final Comparison

In [ ]:
BASELINE_F1  = 0.6254   # TF-IDF LR (confirmed best, EXPERIMENTS_LOG)
BASELINE_AUC = 0.8104
XGB500_F1    = 0.6188   # nb40b XGBoost top-500 + tuning

print('=' * 68)
print('SPECTER vs TF-IDF — FINAL COMPARISON')
print('=' * 68)
print(f'{"Model":<32} {"CV F1":>7} {"Test F1":>8} {"AUC":>7} {"Δ baseline":>10}')
print('-' * 68)
print(f'{"TF-IDF LR (best ever)":<32} {"—":>7} {BASELINE_F1:>8.4f} {BASELINE_AUC:>7.4f} {"baseline":>10}')
print(f'{"XGBoost+feat.sel (nb40b)":<32} {"—":>7} {XGB500_F1:>8.4f} {"0.8112":>7} {XGB500_F1-BASELINE_F1:>+10.4f}')
print('-' * 68)
for name, r in results.items():
    delta = r['test_f1'] - BASELINE_F1
    print(f'{name + " (SPECTER)":<32} {r["cv_f1"]:>7.4f} {r["test_f1"]:>8.4f} {r["auc"]:>7.4f} {delta:>+10.4f}')

best_name = max(results, key=lambda k: results[k]['test_f1'])
best = results[best_name]
print(f'\n*** BEST: {best_name} (SPECTER) — Test F1={best["test_f1"]:.4f} ***')
print(f'    vs TF-IDF baseline: {best["test_f1"] - BASELINE_F1:+.4f}')

## 6. Save Best Model

In [ ]:
best_model = search.best_estimator_  # swap if LR/LGBM wins above
best_threshold = results['XGBoost (tuned)']['threshold']

model_dir = Path('../models/classification')
model_dir.mkdir(parents=True, exist_ok=True)

artifact = {
    'model': best_model,
    'feature_set': 'specter+venue+author',
    'n_features': X_train.shape[1],
    'threshold': best_threshold,
    'test_f1': results['XGBoost (tuned)']['test_f1'],
    'test_auc': results['XGBoost (tuned)']['auc'],
}
with open(model_dir / 'best_specter_model.pkl', 'wb') as f:
    pickle.dump(artifact, f)

print(f"Saved best_specter_model.pkl")
print(f"  Features  : {X_train.shape[1]}")
print(f"  Threshold : {best_threshold:.3f}")
print(f"  Test F1   : {results['XGBoost (tuned)']['test_f1']:.4f}")
print(f"  Test AUC  : {results['XGBoost (tuned)']['auc']:.4f}")